# EfficientNetV2-S: fine-tuning and hyperparameter comparisons

This extends the original architecture notebook: **config list → cached shared `fit()` → comparison → conclusion → optional MLflow**. All experiment and plotting code stays here; `src/` is unchanged.

The registered `efficientnetv2_s` uses **timm `efficientnetv2_rw_s.ra2_in1k`**, a pretrained EfficientNetV2 variant with a one-channel spectrogram stem and two-class head. All layers are fine-tuned from ImageNet weights. Input is the repository's 80-band log-mel representation of 3-second audio; class **1 = allowed**, class **0 = rejected**.

Use **`data/training_dataset_v2` only**, following [DATA_CONTRACT.md](../../../DATA_CONTRACT.md). Preserve its recording-based splits. Do not mix in `our_voices` or the unrelated `sv_*` triplet-learning datasets.

In [ ]:
import hashlib
import json
import math
import os
import sys
import time
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the iml repository.")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from matplotlib.ticker import MaxNLocator
from sklearn.metrics import roc_auc_score, roc_curve

from iml.cli import (check_run_data, data_fingerprint, expand_grid, load_splits,
                     plot_curves, record_run_data, summary_table)
from iml.config import ExperimentConfig
from iml.contract import validate_dataset
from iml.imbalance import imbalance_report
from iml.models import TIMM_IDS, build_model
from iml.runs import eer, load_checkpoint, load_run, run_dir, save_json
from iml.tracking import Tracker, notebook_run
from iml.train import evaluate, fit, get_device

MEMBER = "spdsslg"
MODEL = "efficientnetv2_s"
TOPIC = MODEL
DATA = ROOT / "data" / "training_dataset_v2"  # deliberately ignores IML_DATA
DEVICE = get_device()
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://localhost:5000")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("Dataset:", DATA)
print("Model:", TIMM_IDS[MODEL], "| device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(DEVICE), "| HIP:", torch.version.hip)

## 1. Parameter configurations

The baseline uses pretrained weights, AdamW, LR 3e-4, weight decay 1e-4, dropout 0.2, batch size 32, and cosine scheduling. Every explicit variant changes **one setting** relative to that baseline. This is easier to interpret than changing several settings together; it does not estimate interactions.

The original `EXPLICIT` / `GRID` mechanism remains: clear `EXPLICIT` to use a Cartesian `GRID`. The default is **nine runs × 12 epochs** with a common seed and epoch budget. Seed **42** matches `ExperimentConfig.seed` in `src/iml/config.py`; it has no special statistical meaning. Early stopping is off so epoch comparisons cover the same horizon. Add seeds, e.g. `[42, 1337, 2026]`, to assess training variability; a single-seed difference is exploratory.

All runs use plain, unweighted cross-entropy so their losses have the same definition. `pretrained=True` fine-tunes the full network; `init` is not swept because it would not reinitialize the ImageNet backbone. `augment` is not swept because the current loader does not implement it. ONNX export is disabled during the sweep; it can be done after final model selection.

In [ ]:
BASE = ExperimentConfig(
    model=MODEL, member=MEMBER, topic=TOPIC,
    pretrained=True, loss="ce", class_weight="none",
    optimizer="adamw", lr=3e-4, weight_decay=1e-4,
    dropout=0.2, batch_size=32, scheduler="cosine",
    epochs=12, early_stopping_patience=0, grad_clip=1.0,
    mc_dropout_samples=0, export_onnx=False,
)
SEEDS = [42]
EXPLICIT = [
    [],  # baseline
    ["lr=1e-4"],
    ["lr=1e-3"],
    ["weight_decay=0.01"],
    ["dropout=0.1"],
    ["dropout=0.4"],
    ["batch_size=16"],
    ["scheduler=none"],
    ["optimizer=adam"],
]
VARIED_FIELDS = ("lr", "weight_decay", "dropout", "batch_size", "scheduler", "optimizer")


def variant(cfg):
    return ", ".join(f"{k}={getattr(cfg, k):g}" if isinstance(getattr(cfg, k), float)
                     else f"{k}={getattr(cfg, k)}"
                     for k in VARIED_FIELDS if getattr(cfg, k) != getattr(BASE, k)) or "baseline"


def run_label(cfg):
    return f"{variant(cfg)} | seed={cfg.seed}"


configs = [BASE.with_overrides(*o) for o in EXPLICIT]
CONFIGS = [cfg.with_overrides(f"seed={seed}") for seed in SEEDS for cfg in configs]
assert len({c.run_name() for c in CONFIGS}) == len(CONFIGS), "Remove duplicate configurations."
assert all(c.pretrained and c.loss == "ce" and c.class_weight == "none" for c in CONFIGS)
assert all(not c.finetune_from and not c.mc_dropout_samples for c in CONFIGS)
plan = pd.DataFrame([{"variant": run_label(c), "run_name": c.run_name(),
                      **{k: getattr(c, k) for k in VARIED_FIELDS}, "epochs": c.epochs}
                     for c in CONFIGS])
display(plan)
print(f"{len(CONFIGS)} runs; at most {sum(c.epochs for c in CONFIGS)} epochs total.")

## 2. Validate the data and train (cached)

`check_run_data` protects cached results with a fingerprint of the manifests and clip bytes. Report the class counts and majority/minority ratio for each run. No dataset or split is modified.

Pass **only train and validation loaders to `fit()`**. Validation selects the first epoch attaining the lowest validation EER and supplies the decision threshold. Test predictions are reserved for the final, validation-selected model. Loading/validating test files for the data contract and fingerprint does not use their predictions for selection.

In [ ]:
errors = validate_dataset(DATA)
if errors:
    raise ValueError("Dataset violates the contract:\n" + "\n".join(errors[:20]))
FP = data_fingerprint(DATA)
counts = pd.DataFrame(FP["counts"]).T.rename_axis("split")
counts["total"] = counts.sum(axis=1)
counts["majority_per_minority"] = counts[["allowed", "not_allowed"]].max(axis=1) / counts[["allowed", "not_allowed"]].min(axis=1)
display(counts)
print("Dataset fingerprint:", FP["sha1"])

In [ ]:
records = []
imbalance_rows = []
for cfg in CONFIGS:
    out = run_dir(cfg)
    check_run_data(out, FP, overwrite=False)
    if (out / "metrics.json").exists():
        print("cached:", run_label(cfg), out.name)
        rec = load_run(out)
        n0 = FP["counts"]["train"]["not_allowed"]
        n1 = FP["counts"]["train"]["allowed"]
        report = {"counts": {0: n0, 1: n1}, "ratio": max(n0, n1) / min(n0, n1)}
    else:
        loaders = load_splits(cfg, data_dir=DATA)
        report = imbalance_report(loaders["train"])
        print(run_label(cfg), "train counts:", report["counts"], "ratio:", report["ratio"])
        record_run_data(out, FP)  # persist before fit writes its completion marker
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(DEVICE)
            torch.cuda.synchronize(DEVICE)
        started = time.perf_counter()
        # Original shared fit() and cache flow. No test loader is supplied during tuning.
        rec = fit(cfg, {s: loaders[s] for s in ("train", "val")},
                  device=DEVICE, tracker=Tracker(enabled=False))
        if DEVICE.type == "cuda":
            torch.cuda.synchronize(DEVICE)
        save_json({"fit_wall_s": time.perf_counter() - started,
                   "peak_gpu_memory_mb": torch.cuda.max_memory_allocated(DEVICE) / 2**20
                   if DEVICE.type == "cuda" else None,
                   "device": str(DEVICE), "torch": str(torch.__version__),
                   "hip": torch.version.hip}, out / "timing.json")
        record_run_data(out, FP)
        del loaders
    records.append(rec)
    imbalance_rows.append({"run": rec.name, "variant": run_label(cfg),
                           "rejected": report["counts"][0], "allowed": report["counts"][1],
                           "majority_per_minority": report["ratio"]})

imbalance_table = pd.DataFrame(imbalance_rows)
display(imbalance_table)

## 3. Compare runs using validation results

All rates in the tables are fractions; plots label percentages explicitly. CE is in nats. AUC uses the **continuous probability of allowed=1**, never thresholded decisions. The best epoch and its AUC refer to the same saved `best.pt`, not the epoch with maximum AUC.

The shared history contains per-epoch **validation EER**, online training CE, and evaluation-mode validation CE. Training CE averages minibatches while weights change and dropout is active, so it is not an evaluation-mode training loss. Training EER is available in the shared final report at the best epoch, but is not recorded every epoch; this notebook does not invent missing history.

In [ ]:
table = summary_table(records)  # retained from the original notebook
comparison = ROOT / "notebooks" / MEMBER / "reports" / MODEL
comparison.mkdir(parents=True, exist_ok=True)
table.to_csv(comparison / "table.csv", index=False)
imbalance_table.to_csv(comparison / "imbalance.csv", index=False)

best_predictions = {}
rows = []
for rec in records:
    with np.load(rec.path / "predictions_val.npz", allow_pickle=False) as saved:
        labels, scores = saved["labels"], saved["scores"]
    if set(np.unique(labels)) != {0, 1} or not np.isfinite(scores).all():
        raise ValueError(f"Invalid validation scores for {rec.name}")
    auc = float(roc_auc_score(labels, scores))
    best_predictions[rec.name] = (labels, scores, auc)
    h = rec.history.sort_values("epoch")
    best_idx = np.flatnonzero(h.epoch.to_numpy() == rec.metrics["best_epoch"])[0]
    timing_path = rec.path / "timing.json"
    timing = json.loads(timing_path.read_text()) if timing_path.exists() else {}
    rows.append({"run": rec.name, "variant": variant(rec.cfg), "seed": rec.cfg.seed,
                 **{k: getattr(rec.cfg, k) for k in VARIED_FIELDS},
                 "best_epoch": rec.metrics["best_epoch"], "epochs_run": rec.metrics["epochs_run"],
                 "train_eer_at_best": rec.metrics["train"]["eer"],
                 "val_eer": rec.metrics["val"]["eer"], "val_ce": rec.metrics["val"]["loss"],
                 "val_auc": auc, "val_far": rec.metrics["val"]["far"],
                 "val_frr": rec.metrics["val"]["frr"], "val_macro_f1": rec.metrics["val"]["macro_f1"],
                 "loop_time_s": h.time_s.sum(), "mean_epoch_s": h.time_s.mean(),
                 "time_to_best_s": h.time_s.iloc[:best_idx + 1].sum(),
                 "first_epoch_s": h.time_s.iloc[0],
                 "median_epoch_after_first_s": h.time_s.iloc[1:].median() if len(h) > 1 else np.nan,
                 "fit_wall_s": timing.get("fit_wall_s", np.nan),
                 "peak_gpu_memory_mb": timing.get("peak_gpu_memory_mb", np.nan),
                 "params": rec.metrics["params"]})
results = pd.DataFrame(rows).sort_values(["val_eer", "val_ce", "variant", "seed"])
results.to_csv(comparison / "validation_comparison.csv", index=False)
display(results.drop(columns="run"))

## 4. Plotting: learning curves for every run

Separate plotting cells read saved results; rerun them without retraining. Every figure is both displayed here and saved under `notebooks/spdsslg/reports/efficientnetv2_s/`. Rings and dotted vertical lines mark the validation-selected epoch. Validation FAR/FRR in epoch histories use threshold **0.5**; the final report uses the validation-calibrated threshold.

In [ ]:
import numpy as np
# Plotting cell: one diagnostic dashboard per run.
for rec in records:
    h = rec.history.sort_values("epoch")
    selected = h.loc[h.epoch == rec.metrics["best_epoch"]].iloc[0]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(h.epoch, h.train_loss, label="train CE (online)")
    axes[0].plot(h.epoch, h.val_loss, label="validation CE")
    axes[0].set_ylabel("Cross-entropy (nats)")
    axes[1].plot(h.epoch, 100 * h.val_eer, label="validation EER", color="tab:purple")
    axes[1].scatter(selected.epoch, 100 * selected.val_eer, s=90,
                    facecolors="none", edgecolors="black", zorder=5, label="selected epoch")
    axes[1].set_ylabel("EER (%)")
    axes[2].plot(h.epoch, 100 * h.val_far, label="FAR at threshold 0.5")
    axes[2].plot(h.epoch, 100 * h.val_frr, label="FRR at threshold 0.5")
    axes[2].set_ylabel("Validation error rate (%)")
    for ax in axes:
        ax.axvline(selected.epoch, linestyle=":", color="black", alpha=0.5)
        ax.set_xlabel("Epoch")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(f"{run_label(rec.cfg)} | selected epoch {int(selected.epoch)} | "
                 f"validation AUC={best_predictions[rec.name][2]:.4f}")
    fig.tight_layout()
    plot_path = comparison / f"{rec.name}_epoch_diagnostics.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    display(Image(filename=str(plot_path)))
    plt.close(fig)

In [ ]:

# Plotting cell: overlays make convergence, overfitting, and speed differences visible.
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
colors = plt.cm.tab20(np.linspace(0, 1, len(records)))
for rec, color in zip(records, colors):
    h = rec.history.sort_values("epoch")
    best = h.loc[h.epoch == rec.metrics["best_epoch"]].iloc[0]
    for ax, column, factor in [(axes[0], "train_loss", 1),
                                (axes[1], "val_loss", 1),
                                (axes[2], "val_eer", 100)]:
        ax.plot(h.epoch, factor * h[column], color=color, label=run_label(rec.cfg))
        ax.scatter(best.epoch, factor * best[column], facecolors="none",
                   edgecolors=[color], s=75, zorder=5)
for ax, title in zip(axes.flat, ["Training CE (online)", "Validation CE",
                                  "Validation EER"]):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("EER (%)" if ax is axes[2] else "Cross-entropy (nats)")
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.04), ncol=3, fontsize=8)
fig.tight_layout()
plot_path = comparison / "learning_comparison.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
display(Image(filename=str(plot_path)))
plt.close(fig)

## 5. Plotting: efficiency and learning rate schedule decrease

Timing includes training, validation inference, and data loading per epoch. The first epoch can include expensive GPU kernel selection and warm-up; compare the median epoch time excluding the first epoch as well as the total time. This is observed runtime on this hardware, not an isolated model-compute benchmark. It excludes model initialization and the final report; `fit_wall_s` includes those. All timing comparisons should use the same hardware and execution conditions. Changing batch size also changes optimizer updates per epoch. Identical parameter counts are expected because the architecture stays fixed.

In [ ]:
# Plotting cell: quality versus total/steady runtime, plus learning-rate schedules.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for rec, color in zip(records, colors):
    row = results.loc[results.run == rec.name].iloc[0]
    axes[0].scatter(row.loop_time_s / 60, 100 * row.val_eer, color=color, s=70,
                    label=run_label(rec.cfg))
    axes[1].scatter(row.median_epoch_after_first_s, 100 * row.val_eer, color=color, s=70)
    axes[2].plot(rec.history.epoch, rec.history.lr, color=color)
axes[0].set(xlabel="Total training + validation loop time (minutes)",
            ylabel="Selected validation EER (%)", title="Total runtime, including warm-up")
axes[1].set(xlabel="Median epoch time after first epoch (seconds)",
            ylabel="Selected validation EER (%)", title="Typical epoch cost; lower and left is better")
axes[2].set(xlabel="Epoch", ylabel="Learning rate (log scale)", title="Learning-rate schedules")
axes[2].set_yscale("log")
axes[2].xaxis.set_major_locator(MaxNLocator(integer=True))
for ax in axes:
    ax.grid(alpha=0.25)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8)
fig.tight_layout()
plot_path = comparison / "efficiency_and_lr.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
display(Image(filename=str(plot_path)))
plt.close(fig)

## 6. Plotting: ROC/AUC for each run's selected epoch

**Use validation ROC/AUC at the best validation-EER checkpoint for each run.** A separate ROC for every epoch usually obscures the comparison; the EER and CE histories above already show epoch-by-epoch behavior. AUC here describes ranking, while CE also reflects probability confidence. Best EER, best AUC, and best CE can occur at different epochs.

The left panel overlays all runs. Small multiples below provide a readable individual ROC for each run. The dot uses the shared discrete EER convention. FAR is the false acceptance rate (FPR); the y axis is true acceptance rate (TPR = 1 − FRR). The diagonal is the random-ranking reference. These use continuous scores, so the ROC is not tied to one decision threshold.

Training ROC is optional diagnostic information and usually optimistic. Reserve the **test ROC for the final chosen model**, after freezing the configuration, checkpoint and validation-derived threshold. Do not use test curves to choose hyperparameters. See [scikit-learn ROC documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html) and [validation/test guidance](https://scikit-learn.org/stable/modules/cross_validation.html).

In [ ]:
# Plotting cell: validation ROC overlays, all at the selected checkpoint.
fig, ax = plt.subplots(figsize=(10, 8))
for rec, color in zip(records, colors):
    labels, scores, auc = best_predictions[rec.name]
    fpr, tpr, thresholds = roc_curve(labels, scores, drop_intermediate=False)
    ax.plot(fpr, tpr, color=color,
            label=f"{run_label(rec.cfg)} | epoch {rec.metrics['best_epoch']} | AUC={auc:.3f}")
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="random ranking (AUC=0.5)")
ax.set(xlabel="False acceptance rate (FPR)", ylabel="True acceptance rate (1 − FRR)",
       title="Validation ROC: best validation-EER checkpoint per run", xlim=(0, 1), ylim=(0, 1.02))
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.25)
fig.tight_layout()
plot_path = comparison / "validation_roc_overlay.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
display(Image(filename=str(plot_path)))
plt.close(fig)

In [ ]:
# Plotting cell: one validation ROC panel for each run.
ncols = 3
nrows = math.ceil(len(records) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows), squeeze=False)
for ax, rec, color in zip(axes.flat, records, colors):
    labels, scores, auc = best_predictions[rec.name]
    fpr, tpr, thresholds = roc_curve(labels, scores, drop_intermediate=False)
    idx = np.argmin(np.abs(fpr - (1 - tpr)))
    ax.plot(fpr, tpr, color=color, label=f"AUC={auc:.4f}")
    ax.scatter(fpr[idx], tpr[idx], color="black", s=35,
               label=f"EER≈{100 * rec.metrics['val']['eer']:.2f}%")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set(xlabel="FAR / FPR", ylabel="1 − FRR / TPR", xlim=(0, 1), ylim=(0, 1.02),
           title=f"{run_label(rec.cfg)}\nepoch {rec.metrics['best_epoch']}")
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right", fontsize=8)
for ax in list(axes.flat)[len(records):]:
    ax.set_visible(False)
fig.tight_layout()
plot_path = comparison / "validation_roc_per_run.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
display(Image(filename=str(plot_path)))
plt.close(fig)

## 7. Select on validation; optional final held-out test

Rank variants by mean selected validation EER across seeds, then mean validation CE. For one seed this is the ordinary per-run ranking. Use the smallest predeclared seed as the representative checkpoint, rather than choosing the luckiest seed. Repeated seeds are recommended before treating a small difference as reliable.

The final test cell is **off by default** so rerunning the exploration does not expose test results. Turn it on once the study is finished. It locks the chosen checkpoint hash, dataset fingerprint and validation threshold before evaluating test. FAR/FRR use that unchanged threshold; test EER is a separate descriptive sweep, not a recalibration. A small test split yields a coarse ROC and uncertain estimates; excerpts sharing source recordings are not independent samples.

In [ ]:
ranking = results.groupby("variant", as_index=False).agg(
    mean_val_eer=("val_eer", "mean"), std_val_eer=("val_eer", "std"),
    mean_val_ce=("val_ce", "mean"), mean_val_auc=("val_auc", "mean"), seeds=("seed", "nunique"),
).sort_values(["mean_val_eer", "mean_val_ce", "variant"])
display(ranking)
chosen_variant = ranking.iloc[0].variant
winner = min((r for r in records if variant(r.cfg) == chosen_variant), key=lambda r: r.cfg.seed)
print("Validation-selected variant:", chosen_variant)
print("Representative checkpoint:", winner.best_checkpoint)
print("Best epoch:", winner.metrics["best_epoch"], "| threshold:", winner.metrics["decision_threshold"])

In [ ]:
RUN_FINAL_TEST = True  # enable only after completing validation-based exploration
if RUN_FINAL_TEST:
    current_fp = data_fingerprint(DATA)
    check_run_data(winner.path, current_fp, overwrite=False)
    selection = {"run": winner.name, "best_epoch": winner.metrics["best_epoch"],
                 "data_sha1": current_fp["sha1"],
                 "checkpoint_sha256": hashlib.sha256(winner.best_checkpoint.read_bytes()).hexdigest(),
                 "threshold": winner.metrics["decision_threshold"]}
    lock = comparison / "final_selection.json"
    if lock.exists() and json.loads(lock.read_text()) != selection:
        raise RuntimeError("A different final selection is already locked. Do not retune using test.")
    save_json(selection, lock)
    final_path = winner.path / "final_test.json"
    if final_path.exists():
        saved = json.loads(final_path.read_text())
        if saved["selection"] != selection:
            raise RuntimeError("Saved test results belong to another checkpoint or dataset.")
        final_metrics = saved["metrics"]
    else:
        model = build_model(winner.cfg, load_pretrained=False).to(DEVICE)
        model.load_state_dict(load_checkpoint(winner.best_checkpoint, DEVICE)["state_dict"])
        test_loader = load_splits(winner.cfg, data_dir=DATA)["test"]
        ev = evaluate(model, test_loader, DEVICE, threshold=selection["threshold"])
        np.savez_compressed(winner.path / "predictions_test.npz", labels=ev["_labels"], scores=ev["_scores"])
        final_metrics = {k: v for k, v in ev.items() if not k.startswith("_")}
        final_metrics["auc"] = float(roc_auc_score(ev["_labels"], ev["_scores"])) if len(np.unique(ev["_labels"])) == 2 else float("nan")
        save_json({"selection": selection, "metrics": final_metrics}, final_path)
        del model, test_loader
    display(pd.DataFrame([final_metrics]))
else:
    print("Test set held out. Finish validation analysis before enabling RUN_FINAL_TEST.")

In [ ]:
# Plotting cell: final held-out test ROC, only after the explicitly enabled evaluation above.
if RUN_FINAL_TEST:
    with np.load(winner.path / "predictions_test.npz", allow_pickle=False) as saved:
        labels, scores = saved["labels"], saved["scores"]
    if len(np.unique(labels)) == 2:
        fpr, tpr, _ = roc_curve(labels, scores)
        threshold = winner.metrics["decision_threshold"]
        far = float((scores[labels == 0] >= threshold).mean())
        tar = float((scores[labels == 1] >= threshold).mean())
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.plot(fpr, tpr, label=f"Test AUC={roc_auc_score(labels, scores):.4f}")
        ax.scatter(far, tar, color="black", label=f"Validation threshold={threshold:.3f}")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
        ax.set(xlabel="False acceptance rate (FPR)", ylabel="True acceptance rate (1 − FRR)",
               title=f"Final test ROC: {run_label(winner.cfg)}", xlim=(0, 1), ylim=(0, 1.02))
        ax.grid(alpha=0.25)
        ax.legend(loc="lower right")
        fig.tight_layout()
        plot_path = comparison / "final_test_roc.png"
        fig.savefig(plot_path, dpi=150, bbox_inches="tight")
        display(Image(filename=str(plot_path)))
        plt.close(fig)
    else:
        print("Test ROC/AUC is undefined because the test set contains only one class.")

### Threshold diagnostics and per-clip classifications

These cells inspect **saved predictions from the frozen, validation-selected checkpoint**. They do not retrain, run new model inference, or choose another deployment threshold. They can run independently of the earlier training cells once the final test report exists.

The black dashed line is the validation-selected decision threshold used for every displayed classification. A purple dotted line on the threshold-analysis plots marks the **descriptive test EER threshold only**. Inspecting that point does not change the model's decision rule.

Figures and per-clip CSV tables are saved under `notebooks/spdsslg/reports/efficientnetv2_s/threshold_diagnostics/`, which is ignored by Git. Notebook image outputs are stripped by the repository's existing `nbstripout` clean filter when committed. Numeric and path checks below ensure the saved predictions match the dataset and locked checkpoint.

In [ ]:
# Load saved predictions and retain clip paths, true labels and model decisions.
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from matplotlib.lines import Line2D
from sklearn.metrics import roc_auc_score

from iml.contract import fingerprint
from iml.runs import eer

DIAG_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "pyproject.toml").is_file()), None)
if DIAG_ROOT is None:
    raise RuntimeError("Run this notebook from inside the iml repository.")
DIAG_DATA = DIAG_ROOT / "data" / "training_dataset_v2"
DIAG_REPORTS = DIAG_ROOT / "notebooks" / "spdsslg" / "reports" / "efficientnetv2_s"
DIAG_OUT = DIAG_REPORTS / "threshold_diagnostics"
diag_selection = json.loads((DIAG_REPORTS / "final_selection.json").read_text())
diag_run = DIAG_ROOT / "runs" / "spdsslg" / "efficientnetv2_s" / diag_selection["run"]
diag_test_report = json.loads((diag_run / "final_test.json").read_text())
if diag_test_report["selection"] != diag_selection:
    raise RuntimeError("The test report does not match the locked model selection.")
if fingerprint(DIAG_DATA)["sha1"] != diag_selection["data_sha1"]:
    raise RuntimeError("Dataset changed: clip paths cannot safely be attached to these predictions.")
if hashlib.sha256((diag_run / "best.pt").read_bytes()).hexdigest() != diag_selection["checkpoint_sha256"]:
    raise RuntimeError("Checkpoint changed since the final test evaluation.")

diag_threshold = float(diag_selection["threshold"])
diag_test_eer_threshold = float(diag_test_report["metrics"]["eer_threshold"])
diag_frames = []
diag_summary_rows = []
for diag_split in ("val", "test"):
    with np.load(diag_run / f"predictions_{diag_split}.npz", allow_pickle=False) as saved:
        diag_labels = saved["labels"]
        diag_scores = saved["scores"]
    manifest = pd.read_csv(DIAG_DATA / f"{diag_split}.csv")
    if not np.array_equal(diag_labels, manifest["allowed"].to_numpy()):
        raise RuntimeError(f"{diag_split}: saved label order does not match the manifest.")
    if set(np.unique(diag_labels)) != {0, 1} or not np.isfinite(diag_scores).all():
        raise RuntimeError(f"{diag_split}: finite scores and both classes are required.")
    if not ((diag_scores >= 0) & (diag_scores <= 1)).all():
        raise RuntimeError(f"{diag_split}: scores must be probabilities in [0, 1].")
    predicted = (diag_scores >= diag_threshold).astype(int)
    frame = manifest.copy()
    frame["split"] = diag_split
    frame["score"] = diag_scores
    frame["predicted_allowed"] = predicted
    frame["outcome"] = np.select(
        [(diag_labels == 0) & (predicted == 0),
         (diag_labels == 1) & (predicted == 1),
         (diag_labels == 0) & (predicted == 1)],
        ["Correct reject (TN)", "Correct accept (TP)", "False accept (FP)"],
        default="False reject (FN)",
    )
    frame["assigned_class_score"] = np.where(predicted == 1, diag_scores, 1 - diag_scores)
    frame = frame.sort_values(["score", "path"]).reset_index(drop=True)
    frame["clip_rank"] = np.arange(1, len(frame) + 1)
    diag_frames.append(frame)
    negative = diag_scores[diag_labels == 0]
    positive = diag_scores[diag_labels == 1]
    split_eer, split_eer_threshold = eer(diag_labels, diag_scores)
    diag_summary_rows.append({
        "split": diag_split, "clips": len(frame), "decision_threshold": diag_threshold,
        "FAR (%)": 100 * (negative >= diag_threshold).mean(),
        "FRR (%)": 100 * (positive < diag_threshold).mean(),
        "EER (%)": 100 * split_eer,
        "descriptive_eer_threshold": split_eer_threshold,
        "AUC": roc_auc_score(diag_labels, diag_scores),
        "max_rejected_score": negative.max(), "min_allowed_score": positive.min(),
    })

diag_clips = pd.concat(diag_frames, ignore_index=True)
diag_summary = pd.DataFrame(diag_summary_rows)
diag_errors = diag_clips.loc[diag_clips.allowed != diag_clips.predicted_allowed].sort_values(
    "assigned_class_score", ascending=False,
)
DIAG_OUT.mkdir(parents=True, exist_ok=True)
diag_clips.to_csv(DIAG_OUT / "clip_predictions.csv", index=False)
diag_errors.to_csv(DIAG_OUT / "misclassified_clips.csv", index=False)
diag_summary.to_csv(DIAG_OUT / "threshold_summary.csv", index=False)
print(f"Frozen checkpoint: epoch {diag_selection['best_epoch']} | decision threshold={diag_threshold:.4f}")
print("Assigned-class scores are model outputs, not verified calibrated confidence.")
display(diag_summary.round(6))
print("Misclassified clips, ordered by score assigned to the wrong class:")
display(diag_errors[["split", "clip_rank", "path", "allowed", "predicted_allowed", "score", "outcome"]]
        .style.format({"score": "{:.8f}"}))

In [ ]:
# Plot 1: class score distributions and the validation separation interval.
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)
diag_class_colors = {0: "#3366a3", 1: "#16856b"}
diag_bins = np.linspace(0, 1, 21)
for ax, split in zip(axes, ("val", "test")):
    frame = diag_clips.loc[diag_clips.split == split]
    for label, name in [(0, "Rejected person (true class 0)"), (1, "Allowed person (true class 1)")]:
        scores = frame.loc[frame.allowed == label, "score"].to_numpy()
        ax.hist(scores, bins=diag_bins, weights=np.full(len(scores), 100 / len(scores)),
                color=diag_class_colors[label], alpha=0.35, edgecolor=diag_class_colors[label],
                label=f"{name}; n={len(scores)}")
        # Individual markers expose near-zero/near-one scores hidden by histogram bins.
        ax.plot(scores, np.full(len(scores), 0.025 + label * 0.04), "|",
                color=diag_class_colors[label], markersize=9, transform=ax.get_xaxis_transform())
    row = diag_summary.loc[diag_summary.split == split].iloc[0]
    if row.max_rejected_score < row.min_allowed_score:
        ax.axvspan(row.max_rejected_score, row.min_allowed_score, color="#8cc98c", alpha=0.15,
                    label="Zero-error separation interval")
    ax.axvline(diag_threshold, color="black", linestyle="--", linewidth=1.5,
                label=f"Frozen validation threshold={diag_threshold:.4f}")
    if split == "test":
        ax.axvline(diag_test_eer_threshold, color="#8b4caa", linestyle=":", linewidth=1.5,
                    label=f"Test EER threshold={diag_test_eer_threshold:.4f} (descriptive)")
    ax.set(title=f"{'Validation' if split == 'val' else 'Test'}: true-class score distributions",
           xlabel="Model score for allowed = 1", xlim=(0, 1))
    ax.grid(axis="y", alpha=0.2)
    ax.legend(fontsize=8, loc="upper center")
axes[0].set_ylabel("Clips in bin (% within each true class)")
fig.tight_layout()
diag_plot_path = DIAG_OUT / "class_score_distributions.png"
fig.savefig(diag_plot_path, dpi=160, bbox_inches="tight")
display(Image(filename=str(diag_plot_path)))
plt.close(fig)

In [ ]:
# Plot 2: FAR/FRR at every candidate threshold; test values remain descriptive.
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)
for ax, split in zip(axes, ("val", "test")):
    frame = diag_clips.loc[diag_clips.split == split]
    negative = frame.loc[frame.allowed == 0, "score"].to_numpy()
    positive = frame.loc[frame.allowed == 1, "score"].to_numpy()
    # Include observed scores and just-above-score boundaries to capture discrete changes.
    grid = np.unique(np.r_[np.linspace(0, 1, 1001), frame.score.to_numpy(),
                            np.nextafter(frame.score.to_numpy(), 1.0),
                            diag_threshold, diag_test_eer_threshold])
    far_values = (negative[None, :] >= grid[:, None]).mean(axis=1)
    frr_values = (positive[None, :] < grid[:, None]).mean(axis=1)
    ax.step(grid, 100 * far_values, where="post", label="FAR: rejected people accepted", color="#3366a3")
    ax.step(grid, 100 * frr_values, where="post", label="FRR: allowed people rejected", color="#d67517")
    ax.axvline(diag_threshold, color="black", linestyle="--",
                label=f"Frozen validation threshold={diag_threshold:.4f}")
    row = diag_summary.loc[diag_summary.split == split].iloc[0]
    ax.scatter([diag_threshold, diag_threshold], [row["FAR (%)"], row["FRR (%)"]],
                color=["#3366a3", "#d67517"], s=55, edgecolors="black", zorder=5)
    if split == "val" and row.max_rejected_score < row.min_allowed_score:
        ax.axvspan(row.max_rejected_score, row.min_allowed_score, color="#8cc98c", alpha=0.15,
                    label="Validation zero-error interval")
    if split == "test":
        ax.axvline(diag_test_eer_threshold, color="#8b4caa", linestyle=":",
                    label=f"Test EER threshold={diag_test_eer_threshold:.4f} (descriptive)")
        ax.scatter([diag_test_eer_threshold, diag_test_eer_threshold],
                    [100 * (negative >= diag_test_eer_threshold).mean(),
                     100 * (positive < diag_test_eer_threshold).mean()],
                    color="#8b4caa", marker="D", s=35, zorder=5)
    ax.set(title=f"{'Validation' if split == 'val' else 'Test'}: threshold sensitivity",
           xlabel="Decision threshold", xlim=(0, 1), ylim=(-2, 102))
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Error rate (%)")
fig.tight_layout()
diag_plot_path = DIAG_OUT / "far_frr_vs_threshold.png"
fig.savefig(diag_plot_path, dpi=160, bbox_inches="tight")
display(Image(filename=str(diag_plot_path)))
plt.close(fig)

In [ ]:
# Plot 3: one point per clip, classified at the unchanged validation threshold.
fig, axes = plt.subplots(1, 2, figsize=(15, 11), sharex=True, sharey=True)
diag_styles = {
    "Correct reject (TN)": ("#3366a3", "o"),
    "Correct accept (TP)": ("#16856b", "o"),
    "False accept (FP)": ("#c33c40", "X"),
    "False reject (FN)": ("#d67517", "X"),
}
for ax, split in zip(axes, ("val", "test")):
    frame = diag_clips.loc[diag_clips.split == split]
    for outcome, (color, marker) in diag_styles.items():
        subset = frame.loc[frame.outcome == outcome]
        ax.scatter(subset.score, subset.clip_rank, color=color, marker=marker,
                    s=55 if marker == "X" else 30, alpha=0.9, label=outcome)
    ax.axvline(diag_threshold, color="black", linestyle="--",
                label=f"Frozen threshold={diag_threshold:.4f}")
    errors = frame.loc[frame.allowed != frame.predicted_allowed]
    ax.set(title=f"{'Validation' if split == 'val' else 'Test'}: {len(errors)} mistakes / {len(frame)} clips",
           xlabel="Model score for allowed = 1", xlim=(-0.02, 1.02),
           ylim=(len(frame) + 1, 0))
    ax.set_yticks(np.r_[1, np.arange(5, len(frame) + 1, 5)])
    ax.grid(alpha=0.2)
    # Ring the two class extremes; the exact filenames and scores are listed below.
    extremes = pd.concat([frame.loc[frame.allowed == 0].nlargest(1, "score"),
                          frame.loc[frame.allowed == 1].nsmallest(1, "score")])
    ax.scatter(extremes.score, extremes.clip_rank, s=160, facecolors="none",
                edgecolors="black", linewidths=1.1, zorder=5)
axes[0].set_ylabel("Clip rank within split, sorted by score (see clip_predictions.csv)")
handles = [Line2D([0], [0], marker=marker, linestyle="none", color=color, label=outcome,
                   markersize=7) for outcome, (color, marker) in diag_styles.items()]
handles.append(Line2D([0], [0], color="black", linestyle="--",
                       label=f"Frozen validation threshold={diag_threshold:.4f}"))
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9)
fig.suptitle("Per-clip decisions: true labels versus model predictions; rings mark class extremes")
fig.tight_layout(rect=(0, 0.075, 1, 0.96))
diag_plot_path = DIAG_OUT / "per_clip_classification.png"
fig.savefig(diag_plot_path, dpi=160, bbox_inches="tight")
display(Image(filename=str(diag_plot_path)))
plt.close(fig)

diag_extremes = pd.concat([
    diag_clips.loc[(diag_clips.split == "test") & (diag_clips.allowed == 0)].nlargest(1, "score"),
    diag_clips.loc[(diag_clips.split == "test") & (diag_clips.allowed == 1)].nsmallest(1, "score"),
])
print("Extreme test examples (model scores are not verified calibrated probabilities):")
display(diag_extremes[["clip_rank", "path", "allowed", "predicted_allowed", "score", "outcome"]]
        .style.format({"score": "{:.8f}"}))
print("Figures and complete classification tables saved to:", DIAG_OUT)

### Observations from this execution

All **nine configurations completed 12 epochs**, with seed **42**, on `training_dataset_v2`. The training manifest contains 285 rejected and 228 allowed clips (majority/minority ratio **1.25**). Validation has 63 clips. The validation-selected checkpoint was subsequently evaluated on the separate 63-clip test set.

- Validation-based selection chooses **scheduler=none**, with EER **0.00%**, CE **0.0217**, and AUC **1.0000** at its selected checkpoint. Configuration selection breaks EER ties using validation CE; checkpoint selection within a run retains the first minimum-EER epoch, as the shared trainer specifies.
- **6 of nine runs tie at the minimum validation EER**. The table's CE tie-break gives a reproducible choice, but this small, single-seed validation study does not establish a reliable ranking among those runs.
- The baseline reaches EER **0.00%** at epoch **11**, whereas LR `1e-4` reaches **6.43%** at epoch **5** within the same 12-epoch budget. The lower LR may require a different training horizon; this comparison only supports conclusions within this budget.
- Final test evaluation of the selected **scheduler=none, epoch 9** checkpoint: **EER 20.71%**, **FRR 28.57%** (8/28 allowed clips rejected), and **FAR 17.14%** (6/35 rejected-person clips accepted). FAR and FRR use the unchanged validation-selected threshold **0.5506**; test EER is computed separately by sweeping thresholds and does not recalibrate the model.
- Use the CE curves alongside EER: an unchanged EER can hide changes in probability confidence. EER and CE minima need not coincide, and the evaluated checkpoint is always selected by EER.
- Runtime includes GPU warm-up. Use the timing table and the median-after-first-epoch plot to interpret speed differences; all runs have the same architecture and parameter count.

The gap between validation EER **0.00%** and test EER **20.71%** shows that perfect validation separation did not carry over to the test recordings. Keep the dataset's documented channel/codec differences in mind: apparent voice discrimination can reflect recording conditions; these results do not identify the cause of the gap.

#### Validation threshold versus descriptive test EER threshold

For the selected **scheduler=none, epoch 9** checkpoint, the two reported thresholds have different roles:

| Quantity | Threshold | How it is used |
|---|---:|---|
| Validation-selected decision threshold | **0.5506** | Chosen using validation labels and scores, saved with the checkpoint, then held fixed for test FAR/FRR and inference. |
| Descriptive test EER threshold | **0.3205** | Found by sweeping the already frozen model's test scores to describe its EER operating point. It is not used to retune the checkpoint or replace the decision threshold. |

The shared evaluator searches for the threshold where `abs(FAR - FRR)` is smallest. Its discrete EER estimate is the average of FAR and FRR at that point, so they need not be exactly equal on a small set. This balances the error rates; it does not necessarily minimize total mistakes or application-specific error costs. The threshold is calculated by evaluation code, rather than learned by the optimizer.

On test, the frozen validation threshold gives **FAR 17.14% / FRR 28.57%**. At the descriptive test EER threshold, the rates are **FAR 20.00% / FRR 21.43%**, whose average is **20.71%**. These are different operating points on the same test score distribution.

#### What the separation gap means

A perfect separation gap exists when the highest rejected-person score is below the lowest allowed-person score. Every threshold satisfying `max(rejected scores) < threshold <= min(allowed scores)` makes zero classification errors on that particular set.

For validation, the highest rejected-person score is **0.3119** and the lowest allowed-person score is **0.7893**, giving a gap of approximately **0.4773**. The implementation chooses the midpoint, **0.5506**, when separation is perfect. It is a convenient representative of a whole zero-error interval, not a uniquely established optimal threshold.

Importantly, the descriptive test threshold **0.3205 also lies inside that validation gap** and would also give zero validation errors. Therefore, the numerical difference between the two thresholds alone does not demonstrate distribution shift or identify a problem.

On test there is no such clean separation: a rejected-person clip scores as high as **0.9999995**, while an allowed-person clip scores as low as **0.00014785**. No threshold can classify every test example correctly. The stronger evidence of reduced generalization is **EER rising from 0.00% to 20.71%**, **AUC falling from 1.0000 to 0.8408**, and the overlapping class scores. Changing the threshold cannot repair their ordering or improve AUC.

#### Possible explanations and how to investigate them

The following are hypotheses, not confirmed causes:

- **Limited independent data:** validation and test each contain only 63 clips, and several clips share a source recording. Easy or difficult recordings can substantially affect the observed separation and operating point. Assess uncertainty by resampling whole recordings rather than treating every excerpt as independent.
- **Different recording conditions or example difficulty:** session, microphone, codec, background noise, utterance content, accent, or the mix of negative speakers can change the score distributions. Compare scores and error counts by speaker and source recording, and evaluate new sessions with matched recording conditions for allowed people and strangers.
- **Recording-condition shortcuts or overfitting:** the model may rely on characteristics associated with the labels instead of robust voice identity. [DATA_CONTRACT.md](../../../DATA_CONTRACT.md) documents different recording sources, phones and codecs. This is a plausible risk; the current results do not establish that it caused the test errors.
- **Validation selection optimism:** the same validation set was used to choose epochs and configurations. Favorable validation results can therefore overstate performance on independent recordings. Keep the test results as a report of the frozen selection, rather than repeatedly changing settings to improve them.
- **Unreliable score probabilities:** the softmax scores have not been shown to be calibrated probabilities. Their numerical scale can affect useful thresholds, but a threshold difference alone does not diagnose miscalibration. A reliability diagram can assess calibration; the observed AUC drop also shows reduced ranking performance on test.

A larger validation gap or a higher/lower threshold is not, by itself, proof of a better model. Repeated recordings and broader recording conditions are needed to assess whether the observed behavior is stable.



## 8. Optional MLflow: classification-quality metrics only

The original quality-only logging pattern is retained. Enable it only when you want to log to your configured server. No figures, tables or artifacts are uploaded. Training above disables the tracker to avoid duplicate runs. Validation ROC/AUC remains in this notebook and local artifacts.

In [ ]:
import re

LOG_TO_MLFLOW = False
QUALITY = re.compile(r"^(val|test)_(accuracy|far|frr|eer|macro_f1|f1_macro)$")


def _param(value):
    return str(value) if isinstance(value, (list, tuple, dict)) else value


if LOG_TO_MLFLOW:
    rows_by_run = table.set_index("run").to_dict("index")
    for cfg, rec in zip(CONFIGS, records):
        row = rows_by_run[rec.name]
        quality = {k: float(v) for k, v in row.items() if QUALITY.match(k) and pd.notna(v)}
        params = {"model": MODEL, **{k: _param(v) for k, v in cfg.diff_from_default().items()}}
        with notebook_run(member=MEMBER, topic=TOPIC, run_name=rec.name, params=params,
                          tags={"variant": run_label(cfg)}) as run:
            run.log_metrics(quality)
else:
    print("MLflow logging disabled; all results and plots stay local.")